In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

Process                                                     \
           id Product_Type Shot Velocity_1 Velocity_2 Velocity_3   
0     4207011            2   11      0.156      0.166      0.192   
1     4208012            2   12      0.157      0.166      0.204   
2     4209013            2   13      0.156      0.170      0.204   
3     4210014            2   14      0.154      0.170      0.202   
4     4211015            2   15      0.146      0.160      0.198   
...       ...          ...  ...        ...        ...        ...   
1959  7525657            2  657      0.144      0.173      0.200   
1960  7527658            2  658      0.144      0.173      0.200   
1961  7529659            2  659      0.150      0.166      0.210   
1962  7531660            2  660      0.144      0.174      0.206   
1963  7533661            2  661      0.147      0.174      0.204   

                                                                        ...  \
     High_Velocity Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness  ...   
0            2.723               265           0.012                20  ...   
1            2.730               264           0.014                19  ...   
2            2.715               265           0.012                18  ...   
3            2.717               264           0.011                20  ...   
4            2.684               264           0.012                20  ...   
...            ...               ...             ...               ...  ...   
1959         2.536               264           0.012                17  ...   
1960         2.536               264           0.012                17  ...   
1961         2.492               265           0.011                17  ...   
1962         2.514               264           0.011                16  ...   
1963         2.532               265           0.012                18  ...   

     Defects                                                          \
     Stain_2 Dent_2 Deformation_2 Contamination_2 Impurity_2 Crack_2   
0          0      0             0               0          0       0   
1          0      0             0               0          0       0   
2          0      0             0               0          0       0   
3          0      0             0               0          0       0   
4          0      0             0               0          0       0   
...      ...    ...           ...             ...        ...     ...   
1959       0      0             0               0          0       0   
1960       0      0             0               0          0       0   
1961       0      0             0               0          0       0   
1962       0      0             0               0          0       0   
1963       0      0             0               0          0       0   

                                          Defect_Flag  
     Scratch_2 Buring_Mark_2 Inclusions_2   Is_Defect  
0            0             0            0           0  
1            0             0            0           0  
2            0             0            0           0  
3            0             0            0           0  
4            0             0            0           0  
...        ...           ...          ...         ...  
1959         0             0            0           0  
1960         0             0            0           0  
1961         0             0            0           0  
1962         0             0            0           1  
1963         0             0            0           1  

[1964 rows x 58 columns]

In [ ]:
### 1. 불필요한 id, product_type 컬럼 제거
df = df.drop(columns=[('Process', 'id'), ('Process', 'Product_Type')])

In [ ]:
### 2. 값이 전부 0인 Defects 컬럼 drop
# Defects 하위 컬럼들
defect_cols = df['Defects'].columns

# 값이 전부 0인 Defects 컬럼 찾기
zero_defects = [
    col for col in defect_cols
    if df[('Defects', col)].sum() == 0
]

# 결과 출력
print("삭제될 Defects 컬럼:")
print(zero_defects)

print("\n삭제될 컬럼 수:", len(zero_defects))
print("삭제 전 Defects 컬럼 수:", len(defect_cols))

# 실제 삭제
df = df.drop(columns=[('Defects', col) for col in zero_defects])

print("삭제 후 Defects 컬럼 수:", len(df['Defects'].columns))

삭제될 Defects 컬럼:
['Exfoliation_1', 'Deformation_1', 'Inclusions_1', 'Bubble_2', 'Exfoliation_2', 'Stain_2', 'Deformation_2', 'Scratch_2', 'Buring_Mark_2']

삭제될 컬럼 수: 9
삭제 전 Defects 컬럼 수: 26
삭제 후 Defects 컬럼 수: 17


In [3]:
df['Defects'].sum().sort_values(ascending=False)

Short_Shot_1       176
Blow_Hole_1        112
Stain_1             93
Blow_Hole_2         79
Short_Shot_2        48
Contamination_2      8
Buring_Mark_1        5
Impurity_2           5
Dent_1               4
Contamination_1      4
Dent_2               4
Impurity_1           2
Scratch_1            2
Crack_2              2
Bubble_1             1
Crack_1              1
Inclusions_2         1
dtype: int64

In [7]:
# 불량 범주 정의 (suffix _1, _2 자동 처리)
defect_groups = {
    "표면": [
        "Stain_1",
        "Dent_1",
        "Dent_2",
        "Scratch_1",
        "Buring_Mark_1"
    ],

    "구조": [
        "Short_Shot_1",
        "Short_Shot_2",
        "Blow_Hole_1",
        "Blow_Hole_2",
        "Bubble_1",
        "Crack_1",
        "Crack_2"
    ],

    "이물질": [
        "Contamination_1",
        "Contamination_2",
        "Impurity_1",
        "Impurity_2",
        "Inclusions_2"
    ]
}

In [8]:
import numpy as np
import pandas as pd

defects = df['Defects'].copy()  # columns: Short_Shot_1, Blow_Hole_2, ...

# 각 범주별로 해당하는 defect 컬럼들을 찾아서 "하나라도 1이면 1"로 만들기
y_group = pd.DataFrame(index=df.index)

for group_name, base_names in defect_groups.items():
    # base_names 중 하나로 시작하는 컬럼들 찾기 (예: "Short_Shot" -> "Short_Shot_1", "Short_Shot_2")
    cols = [c for c in defects.columns if any(str(c).startswith(b) for b in base_names)]
    
    if len(cols) == 0:
        # 해당 범주에 매칭되는 컬럼이 없으면 0으로
        y_group[group_name] = 0
    else:
        y_group[group_name] = (defects[cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)

print(y_group.sum().sort_values(ascending=False))   # 범주별 발생 건수 확인
print(y_group.mean().mul(100))                      # 범주별 발생률(%) 확인

구조     395
표면     107
이물질     19
dtype: int64
표면      5.448065
구조     20.112016
이물질     0.967413
dtype: float64


In [9]:
y_group.sum(axis=1).value_counts()

0    1467
1     473
2      24
Name: count, dtype: int64

In [10]:
X = df[['Process','Sensor']].copy()

In [11]:
X.columns = ['_'.join(col) for col in X.columns]

In [12]:
y = y_group

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the f

In [15]:
y_pred = model.predict(X_test)

In [16]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.33      0.08      0.12        13
           1       0.44      0.15      0.22        81
           2       0.00      0.00      0.00         3

   micro avg       0.41      0.13      0.20        97
   macro avg       0.26      0.08      0.12        97
weighted avg       0.42      0.13      0.20        97
 samples avg       0.03      0.03      0.03        97



c:\Users\user\Desktop\project_LSB\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\Desktop\project_LSB\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\Desktop\project_LSB\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", r

In [ ]:
### 3. 남은 defects 중 이상치 확인